In [46]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [47]:
data = load_dataset('jxie/flickr8k')


In [48]:
print(data.shape)
print()
print(data)

{'train': (6000, 6), 'validation': (1000, 6), 'test': (1000, 6)}

DatasetDict({
    train: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 6000
    })
    validation: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
})


In [49]:
train = data['train']
test = data['test']
validation = data['validation']

print(train)
print(train.shape)
print(test.shape)
print(validation.shape)

Dataset({
    features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
    num_rows: 6000
})
(6000, 6)
(1000, 6)
(1000, 6)


In [50]:
train = pd.DataFrame({
    'img': train['image'],
    'caption_0': train['caption_0'],
    'caption_1': train['caption_1'],
    'caption_2': train['caption_2'],
    'caption_3': train['caption_3'],
    'caption_4': train['caption_4'],
})

test = pd.DataFrame({
    'img': test['image'],
    'caption_0': test['caption_0'],
    'caption_1': test['caption_1'],
    'caption_2': test['caption_2'],
    'caption_3': test['caption_3'],
    'caption_4': test['caption_4'],
})

validation = pd.DataFrame({
    'img': validation['image'],
    'caption_0': validation['caption_0'],
    'caption_1': validation['caption_1'],
    'caption_2': validation['caption_2'],
    'caption_3': validation['caption_3'],
    'caption_4': validation['caption_4'],
})


In [51]:
type(train)

pandas.core.frame.DataFrame

### **Text preprocessing**

In [52]:
!pip install cleantext

from cleantext import clean
from tensorflow.keras.layers import TextVectorization

In [53]:
def text_clean(txt):
    txt = clean(
        txt,
        lowercase = True,
        punct = True,
        extra_spaces = True
    )

    return f"<start> {txt} <end>"


def text_clean_2(df):
    for i in ['caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4']:
        df[i] = df[i].map(text_clean)                   # this sends the data sample by sample to the function

    return df


train = text_clean_2(train)                      # this sends the entire dataframe to the function
test = text_clean_2(test)
validation = text_clean_2(validation)

In [54]:
train['caption_0'].shape

(6000,)

In [60]:
train.head(3)

,img,caption_0,caption_1,caption_2,caption_3,caption_4
0,<PIL.JpegImagePlugin.JpegImageFile image mode=...,<start> a black dog is running after a white d...,<start> black dog chasing brown dog through sn...,<start> two dogs chase each other across the s...,<start> two dogs play together in the snow <end>,<start> two dogs running through a low lying b...
1,<PIL.JpegImagePlugin.JpegImageFile image mode=...,<start> a little baby plays croquet <end>,<start> a little girl plays croquet next to a ...,<start> the child is playing croquette by the ...,<start> the kid is in front of a car with a pu...,<start> the little boy is playing with a croqu...
2,<PIL.JpegImagePlugin.JpegImageFile image mode=...,<start> a brown dog in the snow has something ...,<start> a brown dog in the snow holding a pink...,<start> a brown dog is holding a pink shirt in...,<start> a dog is carrying something pink in it...,<start> a dog with something pink in its mouth...


In [57]:
def make_lists(df):
    imgs, captions = [], []
    
    for sample in range(len(df)):
        for i in range(5):
            captions.append(df[f"caption_{i}"].iloc[sample])
            imgs.append(df['img'].iloc[sample])

    return imgs, captions


train_imgs, train_captions = make_lists(train)
test_imgs, test_captions = make_lists(test)
validation_imgs, validation_captions = make_lists(validation)


print(len(train_imgs))
print(len(train_captions))
print()
print(len(test_imgs))
print(len(test_captions))
print()
print(len(validation_imgs))
print(len(validation_captions))

30000
30000

5000
5000

5000
5000


In [58]:
max_len = max(len(i.split()) for i in train_captions)
print(f"CAPTION HODLING MAXIMUM LENGTH: {max_len}")

vectorizer = TextVectorization(
    max_tokens = None,
    output_mode = 'int',               # must keep output mode as int, to remember the order sequence of the words
    output_sequence_length = max_len
)


def do_vectorization(list):
    vectorizer.adapt(list)
    vocab = vectorizer.get_vocabulary()
    vector = vectorizer(list).numpy()

    return vector


train_captions = do_vectorization(train_captions)
test_captions = do_vectorization(test_captions)
validation_captions = do_vectorization(validation_captions)


print()
print(len(train_captions))
print(len(test_captions))
print(len(validation_captions))

CAPTION HODLING MAXIMUM LENGTH: 38

30000
5000
5000


In [59]:
test_captions

array([[ 4,  5, 33, ...,  0,  0,  0],
       [ 4,  5, 33, ...,  0,  0,  0],
       [ 4, 14, 35, ...,  0,  0,  0],
       ...,
       [ 4,  2, 35, ...,  0,  0,  0],
       [ 4,  2, 11, ...,  0,  0,  0],
       [ 4,  2, 11, ...,  0,  0,  0]])

### **Pre-Trained model feature extraction**

In [61]:
import torch as tr
from transformers import ViTImageProcessor, ViTModel

In [67]:
if tr.cuda.is_available():
    device = tr.device('cuda')


processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224')           # for extracting or using the features from model
model = ViTModel.from_pretrained('google/vit-base-patch16-224')                        # for using the model
model = model.to(device)
model.eval()


def img_vectorizer(images):
    img_vector = []
    
    for i in images:
        img = i.convert('RGB')
        inputs = processor(images=img, return_tensors='pt').to(device)

        with tr.no_grad():
            outputs = model(**inputs)

        vec = outputs.pooler_output.detach().cpu().numpy()[0]
        img_vector.append(vec)
        
    return img_vector


train_imgs = img_vectorizer(train['img'])
test_imgs = img_vectorizer(test['img'])
validation_imgs = img_vectorizer(validation['img'])

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [79]:
print(len(train_imgs))
print(len(test_imgs))
print(len(validation_imgs))

len(train_imgs[0])                           # each img is of the shape of 1 x 768 dimensions

6000
1000
1000


768